# Multi-Agent Orchestration, A2A, APIs & Best Practices

## Topics Covered
| # | Section | Concept |
|---|---------|---------|
| 1 | Advanced Workflows | Retry patterns, iteration caps, error recovery |
| 2 | Multi-Agent Orchestration | Supervisor pattern, specialized sub-agents |
| 3 | Agent-to-Agent (A2A) | Communication protocol, message passing |
| 4 | APIs in Agent Workflows | REST calls, JSON-RPC overview inside agents |
| 5 | MCP vs A2A | Side-by-side comparison |
| 6 | Context Sharing | Shared state across agents |
| 7 | Best Practices | Production checklist for scalable agent systems |

> **Pre-requisite:** Run the ENV SETUP cell first before any other cell.

In [ ]:
# ── ENV SETUP ─────────────────────────────────────────────────────────────
import os
import operator
import json
import datetime
import requests
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

load_dotenv(override=True)

llmgw_api_key = os.getenv("LLMGW_API_KEY")
llmgw_base_url = os.getenv("LLMGW_BASE_URL", "https://llmgw-wp.tekstac.com")

if not llmgw_api_key:
    raise ValueError("LLMGW_API_KEY not found. Add it to your .env file.")

os.environ["ANTHROPIC_API_KEY"] = llmgw_api_key
os.environ["ANTHROPIC_BASE_URL"] = llmgw_base_url

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

chat_model = ChatAnthropic(
    model="global.anthropic.claude-sonnet-4-6",
    temperature=0.3,
    anthropic_api_key=llmgw_api_key,
    base_url=llmgw_base_url,
)

print("=" * 55)
print("ENVIRONMENT STATUS")
print("=" * 55)
print(f"LLMGW_API_KEY  : {'✅ set' if llmgw_api_key else '❌ missing'}")
print(f"LLMGW_BASE_URL : {llmgw_base_url}")
print("✅ All imports ready")
print("=" * 55)


# Quick sanity check — confirm Claude responds
_test = chat_model.invoke([HumanMessage(content="Say hello in one word.")])
print(f"   Test call       : {_test.content.strip()}")

---

## Section 1 — Advanced Workflows: Retries, Iteration Caps & Error Recovery

### Why advanced control flow?

A simple linear graph breaks when:
- A tool call fails (network error, bad response)
- The LLM produces invalid output that needs re-generation
- A task needs multiple refinement passes before it's acceptable

LangGraph handles this natively via:

| Pattern | Mechanism | Use case |
|---------|-----------|----------|
| **Retry** | `retry_count` in state + conditional edge back to the same node | Transient failures |
| **Iteration cap** | `iterations` counter — stop after N loops | Prevent infinite loops |
| **Error recovery** | Catch exceptions in a node, write error info to state, route to a fallback node | Graceful degradation |

```
START → [agent] → success? → END
                ↓ fail (retry_count < MAX)
             [agent]   ← back-edge (retry)
                ↓ fail (retry_count == MAX)
             [fallback] → END
```

In [ ]:
print("  Section 1 — Advanced Workflows: Retry + Iteration Cap")
print("=" * 60)

import random

# ── State with retry counter and iteration cap ────────────────────────────
class RetryState(TypedDict):
    messages:    Annotated[list[BaseMessage], operator.add]
    retry_count: int  #Stores how many times the tool was attempted.
    max_retries: int  #Stores the maximum allowed retry attempts.
    iterations:  int  #Stores how many times the agent node has executed.
    max_iter:    int  #Stores the maximum allowed graph iterations.
    error:       str  #Stores the latest error message, if any.

# ── Simulate a flaky tool (fails 60% of the time) ─────────────────────────
@tool
def fetch_live_price(ticker: str) -> str:
    """Fetch the current stock price for a given ticker symbol.""" #tool description.
    if random.random() < 0.6:                    # 60% chance of failure
        raise ConnectionError(f"API timeout for {ticker}")
    prices = {"AAPL": 189.5, "GOOGL": 175.2, "MSFT": 420.8}
    price = prices.get(ticker.upper(), 100.0)
    return f"{ticker.upper()} is trading at ${price}"

flaky_tools   = [fetch_live_price]
tool_map_ft   = {t.name: t for t in flaky_tools}
llm_flaky     = chat_model.bind_tools(flaky_tools)

# ── Nodes ─────────────────────────────────────────────────────────────────
def agent_node(state: RetryState) -> dict:
    resp = llm_flaky.invoke(state["messages"])
    return {"messages": [resp], "iterations": state["iterations"] + 1}

def tools_with_retry(state: RetryState) -> dict:
    """Execute tools; on failure increment retry_count and return error message."""
    last = state["messages"][-1]
    results = []
    for tc in last.tool_calls:
        try:
            output = tool_map_ft[tc["name"]].invoke(tc["args"])
            results.append(ToolMessage(content=str(output), tool_call_id=tc["id"]))
        except Exception as e:
            # Return error as ToolMessage so LLM can decide to retry
            results.append(ToolMessage(
                content=f"ERROR: {e}. Please retry.",
                tool_call_id=tc["id"]
            ))
    return {
        "messages":    results,
        "retry_count": state["retry_count"] + 1,
        "error":       "" if "ERROR" not in results[-1].content else results[-1].content,
    }

def fallback_node(state: RetryState) -> dict:
    """Called when max retries exceeded — return a graceful degradation message."""
    msg = AIMessage(content=(
        f" Could not retrieve live data after {state['max_retries']} attempts. "
        "Please try again later or check your network connection."
    ))
    return {"messages": [msg]}

# ── Routing ───────────────────────────────────────────────────────────────
def route_retry(state: RetryState) -> Literal["tools", "fallback", "__end__"]:
    last = state["messages"][-1]

    if state["iterations"] >= state["max_iter"]:
        print(f"    Iteration cap reached ({state['max_iter']})")
        return END

    if hasattr(last, "tool_calls") and last.tool_calls:
        if state["retry_count"] >= state["max_retries"]:
            print(f"    Max retries ({state['max_retries']}) exceeded → fallback")
            return "fallback"
        return "tools"

    return END

# ── Build Graph ───────────────────────────────────────────────────────────
rb = StateGraph(RetryState)
rb.add_node("agent",    agent_node)
rb.add_node("tools",    tools_with_retry)
rb.add_node("fallback", fallback_node)

rb.add_edge(START, "agent")
rb.add_conditional_edges("agent", route_retry)
rb.add_edge("tools",    "agent")
rb.add_edge("fallback", END)

retry_graph = rb.compile()

# ── Run ───────────────────────────────────────────────────────────────────
print("\n Fetching AAPL stock price (with retry logic)...")
print("─" * 60)

result = retry_graph.invoke({
    "messages":    [HumanMessage(content="What is the current price of AAPL?")],
    "retry_count": 0,
    "max_retries": 4,
    "iterations":  0,
    "max_iter":    10,
    "error":       "",
})

print(f"\n Final: {result['messages'][-1].content}")
print(f"   Retries used   : {result['retry_count']}")
print(f"   Total messages : {len(result['messages'])}")

---

## Section 2 — Multi-Agent Orchestration using LangGraph

### The Supervisor Pattern

Instead of one monolithic agent doing everything, break the work into **specialist sub-agents** coordinated by a **supervisor**:

```
           ┌─────────────────────────────┐
           │         SUPERVISOR          │
           │  (decides who does what)    │
           └──┬────────┬────────┬────────┘
              ▼        ▼        ▼
        [Researcher] [Writer] [Reviewer]
         (facts)     (draft)  (QA check)
```

| Role | Responsibility |
|------|----------------|
| **Supervisor** | Parses the task, assigns sub-agents, aggregates results |
| **Researcher** | Gathers facts / data using tools |
| **Writer** | Drafts content based on researched data |
| **Reviewer** | Validates quality, flags issues |

Each sub-agent is a **node** in the LangGraph. The supervisor uses a **conditional edge** to route to the right node based on a `next_agent` field in shared state.

In [ ]:
print(" Section 2 — Multi-Agent Orchestration (Supervisor Pattern)")
print("=" * 60)

# ── Shared state carries messages + routing signal ────────────────────────
class OrchestratorState(TypedDict):
    messages:   Annotated[list[BaseMessage], operator.add]
    task:       str          # original task
    research:   str          # output from researcher
    draft:      str          # output from writer
    review:     str          # output from reviewer
    next_agent: str          # supervisor routing signal

# ── Sub-Agent Nodes ───────────────────────────────────────────────────────
def researcher_node(state: OrchestratorState) -> dict:
    """Collects facts about the task topic."""
    print("    Researcher working...")
    prompt = [
        SystemMessage(content="You are a research specialist. Provide 3-5 key facts about the given topic. Be concise."),
        HumanMessage(content=f"Research topic: {state['task']}")
    ]
    resp = chat_model.invoke(prompt)
    facts = resp.content.strip()
    print(f"    Research done ({len(facts)} chars)")
    return {
        "messages":   [AIMessage(content=f"[Researcher]: {facts[:120]}...")],
        "research":   facts,
        "next_agent": "writer",
    }

def writer_node(state: OrchestratorState) -> dict:
    """Drafts a short summary using the research."""
    print("     Writer working...")
    prompt = [
        SystemMessage(content="You are a technical writer. Write a 2-paragraph summary using the research provided."),
        HumanMessage(content=f"Research:\n{state['research']}\n\nWrite a clear summary.")
    ]
    resp = chat_model.invoke(prompt)
    draft = resp.content.strip()
    print(f"    Draft ready ({len(draft)} chars)")
    return {
        "messages":   [AIMessage(content=f"[Writer]: Draft created ({len(draft)} chars)")],
        "draft":      draft,
        "next_agent": "reviewer",
    }

def reviewer_node(state: OrchestratorState) -> dict:
    """Reviews the draft and gives a quality verdict."""
    print("    Reviewer working...")
    prompt = [
        SystemMessage(content="You are a quality reviewer. Rate the draft on accuracy and clarity (1-10 each). Be brief."),
        HumanMessage(content=f"Original task: {state['task']}\n\nDraft:\n{state['draft']}\n\nProvide scores and one-line feedback.")
    ]
    resp = chat_model.invoke(prompt)
    review = resp.content.strip()
    print(f"    Review done")
    return {
        "messages":   [AIMessage(content=f"[Reviewer]: {review[:100]}...")],
        "review":     review,
        "next_agent": "DONE",
    }

def supervisor_node(state: OrchestratorState) -> dict:
    """Supervisor: on first call, sets next_agent to researcher."""
    if not state["research"]:
        print("    Supervisor → assigning to Researcher")
        return {"next_agent": "researcher"}
    return {"next_agent": "DONE"}

# ── Routing ───────────────────────────────────────────────────────────────
def supervisor_route(state: OrchestratorState) -> Literal["researcher", "writer", "reviewer", "__end__"]:
    mapping = {
        "researcher": "researcher",
        "writer":     "writer",
        "reviewer":   "reviewer",
    }
    return mapping.get(state["next_agent"], END)

# ── Build Graph ───────────────────────────────────────────────────────────
ob = StateGraph(OrchestratorState)
ob.add_node("supervisor", supervisor_node)
ob.add_node("researcher", researcher_node)
ob.add_node("writer",     writer_node)
ob.add_node("reviewer",   reviewer_node)

ob.add_edge(START,        "supervisor")
ob.add_conditional_edges("supervisor", supervisor_route)
ob.add_edge("researcher", "writer")
ob.add_edge("writer",     "reviewer")
ob.add_edge("reviewer",   END)

orchestrator = ob.compile()

# ── Run ───────────────────────────────────────────────────────────────────
task = "Kubernetes in cloud-native microservices architecture"

print(f"\n Task: {task}")
print("─" * 60)

result = orchestrator.invoke({
    "messages":   [HumanMessage(content=task)],
    "task":       task,
    "research":   "",
    "draft":      "",
    "review":     "",
    "next_agent": "",
})

print("\n" + "=" * 60)
print(" ORCHESTRATION RESULT")
print("=" * 60)
print(f"\n DRAFT:\n{result['draft'][:600]}...")
print(f"\n REVIEW:\n{result['review'][:300]}")

---

## Section 3 — Agent-to-Agent (A2A) Communication Concepts

### What is A2A?

**A2A** is a protocol (proposed by Google) where agents communicate **directly with each other** using structured messages — not via a shared orchestrator.

Each agent exposes an **Agent Card** (like a service manifest) and accepts/sends **Tasks** over HTTP.

```
Agent A ──── HTTP POST (Task) ────► Agent B
             {
               "id": "task-001",
               "type": "research_request",
               "payload": {"topic": "LangGraph"},
               "reply_to": "agent-a.endpoint/results"
             }
Agent A ◄─── HTTP Response (Result) ── Agent B
             {
               "task_id": "task-001",
               "status": "completed",
               "result": {"summary": "..."}
             }
```

### A2A vs Supervisor

| | Supervisor Pattern | A2A Protocol |
|---|---|---|
| **Coordination** | Central orchestrator node | Peer-to-peer direct calls |
| **Coupling** | Tightly coupled via shared state | Loosely coupled via HTTP/JSON |
| **Scalability** | Single process | Independently deployable agents |
| **Standard** | LangGraph-specific | Language/framework agnostic |

In this section we simulate A2A communication **within a single notebook** using Python function calls that mimic the HTTP message exchange.

In [ ]:
print(" Section 3 — Agent-to-Agent (A2A) Communication")
print("=" * 60)

import uuid

# ── A2A Message Schema (mirrors Google A2A spec concepts) ─────────────────
def make_task(task_type: str, payload: dict, sender: str) -> dict:
    """Create a structured A2A task message."""
    return {
        "id":       str(uuid.uuid4())[:8],
        "type":     task_type,
        "sender":   sender,
        "payload":  payload,
        "status":   "pending",
    }

def make_result(task_id: str, result: dict, agent: str) -> dict:
    """Create a structured A2A result message."""
    return {
        "task_id": task_id,
        "agent":   agent,
        "status":  "completed",
        "result":  result,
    }

# ── Agent Card (discovery manifest each agent publishes) ─────────────────
RESEARCH_AGENT_CARD = {
    "name":         "ResearchAgent",
    "version":      "1.0",
    "capabilities": ["research", "fact_check", "summarise"],
    "endpoint":     "agents/research",
}

WRITER_AGENT_CARD = {
    "name":         "WriterAgent",
    "version":      "1.0",
    "capabilities": ["draft", "rewrite", "format"],
    "endpoint":     "agents/writer",
}

# ── Agent Implementations (simulating HTTP handlers) ──────────────────────
def research_agent_handler(task: dict) -> dict:
    """ResearchAgent: handles 'research_request' tasks."""
    topic = task["payload"].get("topic", "AI")
    print(f"\n    ResearchAgent received task [{task['id']}]: research '{topic}'")

    prompt = [
        SystemMessage(content="You are a research agent. Provide 3 concise bullet-point facts."),
        HumanMessage(content=f"Research: {topic}")
    ]
    resp = chat_model.invoke(prompt)
    result = make_result(task["id"], {"facts": resp.content.strip()}, "ResearchAgent")
    print(f"    ResearchAgent → task [{task['id']}] completed")
    return result

def writer_agent_handler(task: dict) -> dict:
    """WriterAgent: handles 'draft_request' tasks."""
    facts = task["payload"].get("facts", "")
    style = task["payload"].get("style", "professional")
    print(f"\n    WriterAgent received task [{task['id']}]: draft in '{style}' style")

    prompt = [
        SystemMessage(content=f"You are a writer agent. Write a short paragraph in a {style} style using the facts provided."),
        HumanMessage(content=f"Facts:\n{facts}")
    ]
    resp = chat_model.invoke(prompt)
    result = make_result(task["id"], {"draft": resp.content.strip()}, "WriterAgent")
    print(f"   WriterAgent → task [{task['id']}] completed")
    return result

# ── Agent Registry (simulates service discovery) ─────────────────────────
AGENT_REGISTRY = {
    "research": research_agent_handler,
    "writer":   writer_agent_handler,
}

def dispatch_task(endpoint_key: str, task: dict) -> dict:
    """Simulate HTTP dispatch — route task to the right agent handler."""
    handler = AGENT_REGISTRY.get(endpoint_key)
    if not handler:
        return {"status": "error", "message": f"No agent at endpoint '{endpoint_key}'"}
    return handler(task)

# ── A2A Workflow: Agent A orchestrates Agent B and Agent C ─────────────────
print("\n A2A Workflow: Coordinator → ResearchAgent → WriterAgent")
print("─" * 60)

topic = "Serverless computing in cloud architectures"
print(f" Topic: {topic}\n")

# Step 1: Coordinator sends research task to ResearchAgent
task1 = make_task("research_request", {"topic": topic}, sender="Coordinator")
print(f" Coordinator → ResearchAgent  |  task_id: {task1['id']}")
research_result = dispatch_task("research", task1)

# Step 2: Coordinator sends draft task to WriterAgent using research output
task2 = make_task(
    "draft_request",
    {"facts": research_result["result"]["facts"], "style": "blog post"},
    sender="Coordinator"
)
print(f"\n Coordinator → WriterAgent    |  task_id: {task2['id']}")
draft_result = dispatch_task("writer", task2)

# ── Final Output ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(" A2A WORKFLOW RESULT")
print("=" * 60)
print(f"\n Research (task {research_result['task_id']}):")
print(research_result["result"]["facts"][:400])
print(f"\n Draft (task {draft_result['task_id']}):")
print(draft_result["result"]["draft"][:500])
print(f"\n Both agents completed via A2A message passing")

---

## Section 4 — APIs in Agent Workflows (REST & JSON-RPC Overview)

### How agents call external APIs

Agents integrate with the outside world through **tools that wrap API calls**. Two dominant styles:

#### REST API (most common)
```
Agent → HTTP GET/POST https://api.example.com/resource
      ← JSON response  {"key": "value"}
```
- Stateless, resource-oriented
- Used by weather APIs, payment gateways, databases-as-a-service, etc.

#### JSON-RPC 2.0 (used by MCP)
```
Agent → POST https://service.com/rpc
        Body: {"jsonrpc":"2.0", "method":"get_weather", "params":{"city":"Bangalore"}, "id":1}
      ← Body: {"jsonrpc":"2.0", "result":{"temp":28}, "id":1}
```
- Method-oriented (not resource-oriented)
- Same endpoint for all calls; method name in body
- Used by MCP, Ethereum nodes, language servers

| | REST | JSON-RPC |
|---|---|---|
| **Endpoint** | One URL per resource | Single endpoint |
| **Verb** | GET, POST, PUT, DELETE | Always POST |
| **Identify action** | HTTP verb + URL | `method` field in body |
| **Error format** | HTTP status codes | `error` field in response body |
| **Used by** | Web APIs, microservices | MCP, blockchain, LSP |

In [ ]:
print(" Section 4 — APIs in Agent Workflows")
print("=" * 60)

# ── Part A: REST API tool (calls JSONPlaceholder — public free test API) ──
@tool
def get_user_info(user_id: int) -> str:
    """Fetch public user profile from REST API by user ID (1-10)."""
    try:
        response = requests.get(
            f"https://jsonplaceholder.typicode.com/users/{user_id}",
            timeout=5
        )
        response.raise_for_status()
        data = response.json()
        return json.dumps({
            "name":    data.get("name"),
            "email":   data.get("email"),
            "company": data.get("company", {}).get("name"),
            "city":    data.get("address", {}).get("city"),
        }, indent=2)
    except Exception as e:
        return f"REST API error: {e}"

@tool
def get_user_posts(user_id: int) -> str:
    """Fetch the number of posts made by a user from REST API."""
    try:
        response = requests.get(
            f"https://jsonplaceholder.typicode.com/posts?userId={user_id}",
            timeout=5
        )
        response.raise_for_status()
        posts = response.json()
        return f"User {user_id} has {len(posts)} posts. Latest: '{posts[0]['title'][:50]}...'"
    except Exception as e:
        return f"REST API error: {e}"

# ── Part B: JSON-RPC message builder (simulated — mirrors MCP format) ─────
def build_jsonrpc_request(method: str, params: dict, req_id: int = 1) -> dict:
    """Build a JSON-RPC 2.0 request object."""
    return {"jsonrpc": "2.0", "method": method, "params": params, "id": req_id}

def build_jsonrpc_response(result: dict, req_id: int = 1) -> dict:
    """Build a JSON-RPC 2.0 success response."""
    return {"jsonrpc": "2.0", "result": result, "id": req_id}

def build_jsonrpc_error(code: int, message: str, req_id: int = 1) -> dict:
    """Build a JSON-RPC 2.0 error response."""
    return {"jsonrpc": "2.0", "error": {"code": code, "message": message}, "id": req_id}

# ── Demonstrate JSON-RPC message flow ─────────────────────────────────────
print("\n─── Part B: JSON-RPC 2.0 Message Format (as used by MCP) ───")

# Simulate a tool call request from an agent to an MCP server
rpc_request = build_jsonrpc_request(
    method="tools/call",
    params={"name": "get_weather", "arguments": {"city": "Bangalore"}},
    req_id=42
)
print(f"\n Agent → MCP Server (Request):")
print(json.dumps(rpc_request, indent=2))

# Simulate a successful response
rpc_response = build_jsonrpc_response(
    result={"content": [{"type": "text", "text": "Bangalore: 28°C, Partly Cloudy"}]},
    req_id=42
)
print(f"\n MCP Server → Agent (Response):")
print(json.dumps(rpc_response, indent=2))

# Simulate an error
rpc_error = build_jsonrpc_error(code=-32601, message="Method not found: get_weather", req_id=43)
print(f"\n MCP Server → Agent (Error):")
print(json.dumps(rpc_error, indent=2))

# ── Part A: Agent using REST API tools ────────────────────────────────────
print("\n─── Part A: LangGraph Agent with REST API Tools ───")

rest_tools   = [get_user_info, get_user_posts]
tool_map_rest = {t.name: t for t in rest_tools}
llm_rest     = chat_model.bind_tools(rest_tools)

class RestState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def rest_agent(state): return {"messages": [llm_rest.invoke(state["messages"])]}
def rest_tools_node(state):
    last, results = state["messages"][-1], []
    for tc in last.tool_calls:
        out = tool_map_rest[tc["name"]].invoke(tc["args"])
        results.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    return {"messages": results}
def rest_route(state):
    last = state["messages"][-1]
    return "tools" if (hasattr(last, "tool_calls") and last.tool_calls) else END

rb2 = StateGraph(RestState)
rb2.add_node("agent", rest_agent)
rb2.add_node("tools", rest_tools_node)
rb2.add_edge(START, "agent")
rb2.add_conditional_edges("agent", rest_route)
rb2.add_edge("tools", "agent")
rest_graph = rb2.compile()

print("\n Calling live REST API (jsonplaceholder.typicode.com)...")
r = rest_graph.invoke({
    "messages": [HumanMessage("Get user info and posts for user ID 3")]
})
print(f"\n Agent: {r['messages'][-1].content.strip()[:500]}")

---

## Section 5 — MCP vs A2A: Standardizing Agent Communication

> **Full MCP coverage** is in `MCP_Session_FIXED.ipynb`. This section focuses on the **comparison** angle.

### The Problem Before Standards

Without standards, every team built custom bridges:

```
Agent ──custom code──► Tool A
Agent ──custom code──► Tool B
Agent ──custom code──► Tool C
```

Result: **N × M integration problem** — N agents × M tools = N×M custom connectors.

### The Two Standards

```
┌─────────────────────────────────────────────────────┐
│              MCP (Model Context Protocol)            │
│  Anthropic (2024)  —  Agent ↔ Tools/Data Sources   │
│  Protocol: JSON-RPC 2.0 over stdio / HTTP SSE       │
│  Goal: Standardize how LLMs access external tools   │
└─────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────┐
│              A2A (Agent-to-Agent Protocol)           │
│  Google (2025)  —  Agent ↔ Agent                   │
│  Protocol: JSON over HTTP  (Agent Cards + Tasks)    │
│  Goal: Standardize how agents discover & talk       │
└─────────────────────────────────────────────────────┘
```

| Dimension | MCP | A2A |
|-----------|-----|-----|
| **Purpose** | LLM ↔ Tools/Resources | Agent ↔ Agent |
| **Created by** | Anthropic | Google |
| **Transport** | JSON-RPC 2.0 (stdio / HTTP SSE) | JSON over HTTP |
| **Discovery** | Tool schemas (`tools/list`) | Agent Cards (published manifests) |
| **Message unit** | Tool call + result | Task + result |
| **Auth** | API keys, OAuth | OAuth 2.0 |
| **State** | Stateless tool calls | Stateful tasks with streaming |
| **Best for** | Single-agent tool access | Multi-agent systems |
| **Complements** | Works with A2A | Works with MCP |

### They Are Complementary, Not Competing

```
[Agent A] ──A2A──► [Agent B]
                        │
                       MCP
                        │
                        ▼
                  [Database Tool]
                  [Web Search Tool]
                  [Calendar Tool]
```

Agent B uses **A2A** to receive tasks from Agent A, and **MCP** to access its tools.

---

## Section 6 — Context Sharing Across Agents and Systems

### Why context sharing matters

Each agent in a multi-agent system needs to know what others have done. Without sharing:
- Agent B re-does work Agent A already completed
- Agents contradict each other
- The final output lacks coherence

### Patterns for sharing context

| Pattern | How | When to use |
|---------|-----|-------------|
| **Shared state dict** | All agents read/write one `TypedDict` | Same process, LangGraph workflow |
| **Message history** | `Annotated[list, operator.add]` accumulates all messages | Traceability matters |
| **Checkpointed memory** | `MemorySaver` / `SqliteSaver` | Cross-session persistence |
| **External store** | Redis, PostgreSQL, vector DB | Multi-process / distributed agents |

In [ ]:
print(" Section 6 — Context Sharing Across Agents")
print("=" * 60)

# ── Shared context state — all agents read AND write ──────────────────────
class SharedContextState(TypedDict):
    messages:    Annotated[list[BaseMessage], operator.add]
    user_name:   str
    user_goal:   str
    plan:        str          # planner writes, executor reads
    execution:   str          # executor writes, summariser reads
    summary:     str          # summariser writes

# ── Agent 1: Planner — reads goal, writes plan ────────────────────────────
def planner_agent(state: SharedContextState) -> dict:
    print(f"    Planner: creating plan for '{state['user_name']}'")
    prompt = [
        SystemMessage(content="You are a project planner. Create a 3-step action plan. Be concise."),
        HumanMessage(content=f"User: {state['user_name']}\nGoal: {state['user_goal']}")
    ]
    resp = chat_model.invoke(prompt)
    plan = resp.content.strip()
    print(f"    Planner → plan written ({len(plan)} chars)")
    return {
        "messages": [AIMessage(content=f"[Planner] Plan created for {state['user_name']}")],
        "plan":     plan,
    }

# ── Agent 2: Executor — reads plan, writes execution notes ───────────────
def executor_agent(state: SharedContextState) -> dict:
    print(f"     Executor: acting on plan for '{state['user_name']}'")
    prompt = [
        SystemMessage(content="You are an executor. Given a plan, list what was done in each step. Be brief."),
        HumanMessage(content=f"User goal: {state['user_goal']}\n\nPlan:\n{state['plan']}\n\nDescribe execution of each step.")
    ]
    resp = chat_model.invoke(prompt)
    execution = resp.content.strip()
    print(f"    Executor → execution notes written ({len(execution)} chars)")
    return {
        "messages":  [AIMessage(content="[Executor] Steps executed")],
        "execution": execution,
    }

# ── Agent 3: Summariser — reads ALL context, writes final summary ─────────
def summariser_agent(state: SharedContextState) -> dict:
    print(f"    Summariser: consolidating context for '{state['user_name']}'")
    prompt = [
        SystemMessage(content="You are a report writer. Create a one-paragraph executive summary."),
        HumanMessage(content=(
            f"User: {state['user_name']}\n"
            f"Goal: {state['user_goal']}\n\n"
            f"Plan:\n{state['plan']}\n\n"
            f"Execution:\n{state['execution']}"
        ))
    ]
    resp = chat_model.invoke(prompt)
    summary = resp.content.strip()
    print(f"    Summariser → final summary written")
    return {
        "messages": [AIMessage(content="[Summariser] Summary complete")],
        "summary":  summary,
    }

# ── Build graph ───────────────────────────────────────────────────────────
cb = StateGraph(SharedContextState)
cb.add_node("planner",    planner_agent)
cb.add_node("executor",   executor_agent)
cb.add_node("summariser", summariser_agent)

cb.add_edge(START,       "planner")
cb.add_edge("planner",   "executor")
cb.add_edge("executor",  "summariser")
cb.add_edge("summariser", END)

context_graph = cb.compile()

# ── Run ───────────────────────────────────────────────────────────────────
print("\n Running 3-agent pipeline with shared context...")
print("─" * 60)

result = context_graph.invoke({
    "messages":  [HumanMessage("Start workflow")],
    "user_name": "Ananya",
    "user_goal": "Deploy a FastAPI application to AWS ECS with CI/CD",
    "plan":      "",
    "execution": "",
    "summary":   "",
})

print("\n" + "=" * 60)
print(" SHARED CONTEXT RESULT")
print("=" * 60)
print(f"\n PLAN (written by Planner, read by Executor + Summariser):")
print(result["plan"][:500])
print(f"\n EXECUTIVE SUMMARY (written by Summariser using full context):")
print(result["summary"][:500])
print(f"\n Message trace: {len(result['messages'])} messages in shared history")

---

## Section 7 — Best Practices for Scalable Agent Systems

### 1. Design State Carefully

```python
# ✅ Use TypedDict with clear field names
class AgentState(TypedDict):
    messages:    Annotated[list[BaseMessage], operator.add]
    retry_count: int       # always initialise to 0
    error:       str       # always initialise to ""

# ❌ Avoid storing large blobs (full documents) in state — use IDs + external store
```

### 2. Always Cap Retries and Iterations

```python
# ✅ Guard against infinite loops
MAX_RETRIES = 3
MAX_ITER    = 10

def route(state):
    if state["retry_count"] >= MAX_RETRIES: return "fallback"
    if state["iterations"]  >= MAX_ITER:    return END
    ...
```

### 3. Use Specialized Sub-Agents, Not One Giant Agent

| ❌ Anti-pattern | ✅ Best practice |
|----------------|-----------------|
| One agent with 15 tools | Supervisor routes to 3-4 focused agents |
| Prompt with 2000 tokens of instructions | Each agent has a clear, short system prompt |
| Single LLM call does research + write + review | Separate nodes for each concern |

### 4. Standardize Tool Interfaces

```python
# ✅ Every tool returns a string (LangChain ToolMessage expects str)
@tool
def my_tool(input: str) -> str:
    result = do_something(input)
    return json.dumps(result)   # always serialise complex objects
```

### 5. Prefer MCP for Tool Interoperability

- Define tools as **MCP servers** so any agent framework (LangChain, CrewAI, AutoGen) can use them
- Use **JSON-RPC 2.0** message format — it's language-agnostic
- Publish **Tool Schemas** (JSON Schema) so agents can auto-discover tools

### 6. Use Checkpointers for Production Memory

| Checkpointer | Use case | Persistence |
|---|---|---|
| `MemorySaver` | Dev / single process | In-memory only |
| `SqliteSaver` | Single server prod | File on disk |
| `PostgresSaver` | Multi-instance prod | External DB |

### 7. Observability Checklist

- [ ] Every agent node logs `node_name`, `input_keys`, `output_keys`
- [ ] Retry attempts are logged with error details
- [ ] LangSmith tracing enabled (`LANGCHAIN_TRACING_V2=true`)
- [ ] Max tokens per LLM call is bounded
- [ ] Sensitive data is redacted before logging

### 8. Security for Agent APIs

- [ ] Validate all tool input args at the boundary (not just inside the tool)
- [ ] Never pass raw user input directly to SQL/shell commands
- [ ] Rate-limit external API tool calls
- [ ] Use scoped API keys per agent (principle of least privilege)
- [ ] HITL (`interrupt_before`) for any destructive or financial action